# 🎯 GECA Brand Detection — Entrenamiento YOLO26
Notebook para entrenar modelos de detección de marcas.

In [ ]:
!pip install ultralytics mlflow -q

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Verificar Dataset

In [ ]:
import os, yaml

# ============================================
# CONFIGURA AQUI
# ============================================
DATASET_NAME = 'Bu_Bublik_v2'
EXPERIMENT_NAME = f'{DATASET_NAME}_v1'

SHARED = '/home/jovyan/shared'
DATASET_PATH = f'{SHARED}/datasets/ready/{DATASET_NAME}'
YAML_PATH = f'{DATASET_PATH}/data.yaml'
MODEL_BASE = f'{SHARED}/models/yolo26m.pt'
MODELS_OUTPUT = f'{SHARED}/models'
RUNS_DIR = f'{SHARED}/runs/detect'

assert os.path.exists(YAML_PATH), f'data.yaml no encontrado: {YAML_PATH}'
assert os.path.exists(MODEL_BASE), f'Modelo base no encontrado: {MODEL_BASE}'

with open(YAML_PATH) as f:
    data_config = yaml.safe_load(f)
names = data_config['names']

print(f'Dataset: {DATASET_NAME}')
print(f'Clases ({data_config["nc"]}): {names}')
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_PATH, 'images', split)
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)
    n_imgs = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    n_lbls = len(os.listdir(lbl_dir)) if os.path.isdir(lbl_dir) else 0
    print(f'  {split}: {n_imgs} images, {n_lbls} labels')

## 3. Visualizar Muestras

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

train_imgs = os.path.join(DATASET_PATH, 'images', 'train')
train_lbls = os.path.join(DATASET_PATH, 'labels', 'train')
colors = plt.cm.tab10.colors

img_files = [f for f in os.listdir(train_imgs) if f.endswith(('.png', '.jpg'))]
samples = random.sample(img_files, min(4, len(img_files)))

fig, axes = plt.subplots(1, len(samples), figsize=(5*len(samples), 5))
if len(samples) == 1: axes = [axes]

for ax, img_name in zip(axes, samples):
    img = Image.open(os.path.join(train_imgs, img_name))
    w, h = img.size
    ax.imshow(img)
    lbl_name = img_name.rsplit('.', 1)[0] + '.txt'
    lbl_path = os.path.join(train_lbls, lbl_name)
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    xc, yc, bw, bh = float(parts[1])*w, float(parts[2])*h, float(parts[3])*w, float(parts[4])*h
                    rect = patches.Rectangle((xc-bw/2, yc-bh/2), bw, bh, linewidth=2, edgecolor=colors[cid % len(colors)], facecolor='none')
                    ax.add_patch(rect)
                    cn = names[cid] if isinstance(names, dict) and cid in names else str(cid)
                    ax.text(xc-bw/2, yc-bh/2-5, cn, color=colors[cid%len(colors)], fontsize=8, fontweight='bold', backgroundcolor='black')
    ax.set_title(img_name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Distribución de Clases

In [ ]:
from collections import Counter

class_counts = Counter()
for split in ['train', 'val', 'test']:
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)
    if not os.path.isdir(lbl_dir): continue
    for f in os.listdir(lbl_dir):
        if not f.endswith('.txt'): continue
        with open(os.path.join(lbl_dir, f)) as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    cn = names[cid] if isinstance(names, dict) and cid in names else str(cid)
                    class_counts[cn] += 1

labels_list, counts = zip(*class_counts.most_common())
plt.figure(figsize=(10, max(3, len(labels_list)*0.5)))
plt.barh(labels_list, counts, color='#6c5ce7')
plt.xlabel('Anotaciones')
plt.title('Distribución de Clases')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
for c, n in class_counts.most_common(): print(f'  {c}: {n}')

## 5. Entrenar YOLO26

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_BASE)

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    patience=20,
    name=EXPERIMENT_NAME,
    project=RUNS_DIR,
    exist_ok=True,
    plots=True,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
)

## 6. Resultados del Entrenamiento

In [ ]:
from IPython.display import Image as IPImage, display

results_dir = f'{RUNS_DIR}/{EXPERIMENT_NAME}'

for plot in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
             'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png']:
    plot_path = os.path.join(results_dir, plot)
    if os.path.exists(plot_path):
        print(f'\n{"="*50}')
        print(f'  {plot}')
        print(f'{"="*50}')
        display(IPImage(filename=plot_path, width=800))

## 7. Validar Modelo

In [ ]:
best_model_path = f'{RUNS_DIR}/{EXPERIMENT_NAME}/weights/best.pt'
best_model = YOLO(best_model_path)

metrics = best_model.val()

print(f'\nmAP50-95: {metrics.box.map:.4f}')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP75:    {metrics.box.map75:.4f}')
print(f'\nPer-class mAP50-95:')
for i, m in enumerate(metrics.box.maps):
    cn = names[i] if isinstance(names, dict) and i in names else str(i)
    print(f'  {cn}: {m:.4f}')

## 8. Predicciones Visuales

In [ ]:
test_imgs = os.path.join(DATASET_PATH, 'images', 'test')
if not os.path.isdir(test_imgs) or len(os.listdir(test_imgs)) == 0:
    test_imgs = os.path.join(DATASET_PATH, 'images', 'val')

test_files = [os.path.join(test_imgs, f) for f in os.listdir(test_imgs) if f.endswith(('.png','.jpg'))][:6]
results_pred = best_model(test_files, conf=0.25)

n = min(6, len(results_pred))
cols = min(3, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
if n == 1: axes = [axes]
elif rows > 1: axes = axes.flat
for ax, result in zip(axes, results_pred):
    img = result.plot()
    ax.imshow(img[:, :, ::-1])
    ax.set_title(f'{len(result.boxes)} detecciones', fontsize=10)
    ax.axis('off')
plt.suptitle('Predicciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Guardar Modelo

In [ ]:
import shutil

MODEL_NAME = f'geca_{EXPERIMENT_NAME}_best.pt'
output_path = os.path.join(MODELS_OUTPUT, MODEL_NAME)
best_model_path = f'{RUNS_DIR}/{EXPERIMENT_NAME}/weights/best.pt'

shutil.copy(best_model_path, output_path)
print(f'\n✓ Modelo guardado en: {output_path}')
print(f'  Tamaño: {os.path.getsize(output_path) / 1e6:.1f} MB')
print(f'\n  Para Inference notebook: MODEL_PATH = "{output_path}"')

## 10. Registrar en MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri('http://geca_mlflow:5000')
mlflow.set_experiment('GECA_Brand_Detection')

with mlflow.start_run(run_name=EXPERIMENT_NAME):
    mlflow.log_param('model_base', 'yolo26m')
    mlflow.log_param('dataset', DATASET_NAME)
    mlflow.log_param('epochs', 100)
    mlflow.log_param('batch', 32)
    mlflow.log_param('imgsz', 640)
    mlflow.log_param('num_classes', data_config['nc'])
    mlflow.log_metric('mAP50-95', metrics.box.map)
    mlflow.log_metric('mAP50', metrics.box.map50)
    mlflow.log_metric('mAP75', metrics.box.map75)
    mlflow.log_artifact(best_model_path)
    for plot in ['results.png', 'confusion_matrix.png']:
        p = os.path.join(results_dir, plot)
        if os.path.exists(p): mlflow.log_artifact(p)
    print(f'\n✓ Registrado en MLflow')
    print(f'  Ver en: http://10.43.13.186:5000')